In [1]:
from numba import cuda, float32
import numpy as np

In [2]:
N = 4

A = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 1, 2, 3],
              [4, 5, 6, 7]], dtype=np.float32)

B = np.array([[1, 0, 1, 0],
              [0, 1, 0, 1],
              [1, 1, 1, 1],
              [0, 0, 1, 1]], dtype=np.float32)

print("Matrix A:")
print(A)

print("\nMatrix B:")
print(B)

Matrix A:
[[1. 2. 3. 4.]
 [5. 6. 7. 8.]
 [9. 1. 2. 3.]
 [4. 5. 6. 7.]]

Matrix B:
[[1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [1. 1. 1. 1.]
 [0. 0. 1. 1.]]


In [3]:
@cuda.jit
def matrix_mul_shared_local(A, B, C):

    # Shared Memory
    sA = cuda.shared.array((16, 16), dtype=float32)
    sB = cuda.shared.array((16, 16), dtype=float32)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = cuda.blockIdx.y * cuda.blockDim.y + ty
    col = cuda.blockIdx.x * cuda.blockDim.x + tx

    # Local Memory Variable
    temp = 0.0

    if row < A.shape[0] and col < B.shape[1]:

        # Load matrices into Shared Memory
        sA[ty, tx] = A[row, tx]
        sB[ty, tx] = B[ty, col]

        cuda.syncthreads()

        # Matrix Multiplication
        for k in range(A.shape[1]):
            temp += sA[ty, k] * sB[k, tx]

        C[row, col] = temp

In [4]:
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)

d_C = cuda.device_array((N, N), dtype=np.float32)

In [5]:
threads_per_block = (N, N)
blocks_per_grid = (1, 1)

matrix_mul_shared_local[blocks_per_grid, threads_per_block](
    d_A, d_B, d_C
)

d:\Python\lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [6]:
C = d_C.copy_to_host()

print("Result Matrix:")
print(C)

Result Matrix:
[[ 4.  5.  8.  9.]
 [12. 13. 20. 21.]
 [11.  3. 14.  6.]
 [10. 11. 17. 18.]]


In [7]:
print("NumPy Result:")
print(np.dot(A, B))

NumPy Result:
[[ 4.  5.  8.  9.]
 [12. 13. 20. 21.]
 [11.  3. 14.  6.]
 [10. 11. 17. 18.]]
